# Sistema de PLN — Muestreo y clasificación en GPU sobre *Ecommerce Text Classification*

**Alumno:** Víctor Manuel Santos Martínez
**Materia:** Cómputo de Alto Desempeño
**Actividad:** PLN (Muestreo) — Corte 2 · Bitácora 04
**Entorno:** Google Colab (GPU NVIDIA T4) · RAPIDS cuML · imbalanced-learn
**Repositorio:** `MIA_C3_CAD_Corte_02_Bitacora_04`

---

El notebook implementa un **pipeline de PLN de tres fases**, siguiendo el diagrama de la
actividad (CPU en paralelo → GPU en secuencia):

1. **Fase 1 · CPU en paralelo** — extracción de características: *keywords* con `TfidfVectorizer`,
   sentimiento con `TextBlob`, entidades con `spaCy`. Un proceso por técnica
   (`ProcessPoolExecutor`, patrón *Strategy* en `feature_extractors.py`).
2. **Fase 2 · CPU en paralelo** — **muestreo** para balancear las 4 clases: submuestreo aleatorio,
   sobremuestreo con réplicas y sobremuestreo sintético **SMOTE**. Un proceso por técnica
   (`ProcessPoolExecutor`, patrón *Strategy* en `sampling_strategies.py`).
3. **Fase 3 · GPU en secuencia** — clasificación con **Random Forest**, **Regresión Logística** y
   **SVM** (cuML) sobre cada conjunto muestreado y sobre el *baseline* sin balancear
   (`classification_models_gpu.py`).


## Contexto y pregunta guía

El conjunto *Ecommerce Text Classification* contiene ~50,000 descripciones de producto en 4 categorías
(**Household**, **Books**, **Electronics**, **Clothing & Accessories**). Las clases están **desbalanceadas**:
*Household* aporta el 38 % de las instancias y *Clothing & Accessories* apenas el 17 % (razón ≈ 2.2 : 1).

En un problema desbalanceado, la **exactitud (accuracy) engaña**: un clasificador puede alcanzar un accuracy
alto limitándose a predecir bien la clase mayoritaria mientras ignora a las minoritarias. Por eso, además del
accuracy, este estudio mide **balanced accuracy** (media del *recall* por clase) y **macro-F1** (media del F1
por clase), que sí penalizan el descuido de las clases pequeñas.

> **Pregunta guía.** ¿Corrige el muestreo (submuestreo / sobremuestreo / SMOTE) el sesgo hacia la clase
> mayoritaria? ¿Qué técnica mejora más el desempeño en las clases minoritarias (macro-F1) frente al *baseline*
> sin balancear, y a qué costo de tiempo? Además, en clave de Cómputo de Alto Desempeño: ¿cuánto se acelera la
> etapa de muestreo al ejecutar las tres técnicas **en paralelo** sobre CPU?


## 0. Entorno: Colab GPU + RAPIDS + imbalanced-learn

> **Antes de ejecutar:** en Colab, `Entorno de ejecución → Cambiar tipo de entorno → GPU (T4)`.

Se respeta la organización de Drive de la Actividad 3 (dentro de `MyDrive/Cuatrimestre 3/Cómputo de Alto
Desempeño/Segundo parcial/`):

- **`Datasets/`** — datos compartidos. Ya contiene `ecommerceDataset.csv` (dataset crudo, 2 columnas), de
  donde se lee la entrada.
- **`Resultados_Actividad_4/`** — carpeta de esta actividad. Aloja los tres módulos `.py`
  (`feature_extractors.py`, `sampling_strategies.py`, `classification_models_gpu.py`) y las salidas
  (`results/`, `resultados_matrices/`, `figuras/`). Sus módulos se importan con prioridad en `sys.path`, de
  modo que **no** interfieren con el `classification_models_gpu.py` de la Actividad 3 que vive en `Datasets/`.

> ⚠️ **RAPIDS en Colab.** Se instala con el instalador oficial `rapidsai-csp-utils`, que valida la GPU e
> instala la versión de RAPIDS compatible con el driver/CUDA actual de Colab. El `pip install` directo de
> `cudf-cu12 cuml-cu12` puede traer ruedas más nuevas que el driver y provoca `cudaErrorInsufficientDriver` al
> primer uso de la GPU. La instalación tarda unos minutos y **puede reiniciar la sesión**; si ya ejecutaste una
> instalación previa, usa `Entorno de ejecución → Reiniciar sesión` antes de volver a ejecutar todo.


In [ ]:
# --- RAPIDS (cuDF/cuML) compatibles con el driver de Colab ---
# Instalador oficial: valida la GPU e instala la version de RAPIDS que coincide con el
# CUDA/driver actual de Colab. Evita 'cudaErrorInsufficientDriver', que ocurre cuando el
# pip directo de cudf-cu12/cuml-cu12 trae ruedas mas nuevas que el driver del runtime.
!git clone https://github.com/rapidsai/rapidsai-csp-utils.git
!python rapidsai-csp-utils/colab/pip-install.py


In [ ]:
# --- Dependencias de CPU (extraccion y muestreo) ---
# Sin -U para no alterar las versiones de numpy/pandas/scikit-learn que fija RAPIDS.
!pip install -q imbalanced-learn textblob tabulate spacy
!python -m spacy download en_core_web_sm -q


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import sys
from pathlib import Path

# Misma organización que la Actividad 3: 'Datasets' (datos compartidos) y 'Resultados_Actividad_N' (salidas).
SEGUNDO_PARCIAL = Path('/content/drive/MyDrive/Cuatrimestre 3/Cómputo de Alto Desempeño/Segundo parcial')
DATA_DIR = SEGUNDO_PARCIAL / 'Datasets'                 # dataset crudo (ya presente)
BASE     = SEGUNDO_PARCIAL / 'Resultados_Actividad_4'   # módulos .py de esta actividad + salidas

# Prioridad en sys.path: usa el classification_models_gpu.py de esta actividad, no el de la Act. 3 en Datasets.
sys.path.insert(0, str(BASE))

RAW_CSV     = DATA_DIR / 'ecommerceDataset.csv'
RESULTS_DIR = BASE / 'results'
MATRIX_DIR  = BASE / 'resultados_matrices'
FIG_DIR     = BASE / 'figuras'
for d in (RESULTS_DIR, MATRIX_DIR, FIG_DIR):
    d.mkdir(parents=True, exist_ok=True)

RANDOM_STATE = 42
TEST_SIZE = 0.20
MAX_TFIDF_FEATURES = 10_000   # dim. moderada: deja margen de RAM/VRAM para el RF denso sobre datos sobremuestreados

assert RAW_CSV.exists(), f'No se encontró el dataset en {RAW_CSV}'
for m in ('feature_extractors.py', 'sampling_strategies.py', 'classification_models_gpu.py'):
    assert (BASE / m).exists(), f'Falta el módulo {m} en {BASE}'
print('Datos :', RAW_CSV)
print('Base  :', BASE)


## 1. Fase CPU paralela — Extracción de características

Se reutiliza `feature_extractors.py` (linaje Bitácora 01/02). Las tres técnicas — *keywords* TF-IDF,
sentimiento (TextBlob) y entidades (spaCy) — se ejecutan **en paralelo**, una por proceso, con
`concurrent.futures.ProcessPoolExecutor(max_workers=3)`. Cada objeto estrategia es *picklable* por vivir en un
módulo importable, requisito para despacharlo a un proceso trabajador. Medimos el tiempo por técnica y el
**tiempo de pared** para cuantificar el *speedup* del paralelismo a nivel de proceso.


In [ ]:
import pandas as pd

raw = pd.read_csv(RAW_CSV, header=None, names=['category', 'description'])
raw = raw.dropna().drop_duplicates().reset_index(drop=True)
raw['description'] = raw['description'].astype(str)
print('Instancias:', len(raw))
raw['category'].value_counts()


In [ ]:
import time
from concurrent.futures import ProcessPoolExecutor
from feature_extractors import DEFAULT_EXTRACTORS, run_extractor

texts = raw['description'].tolist()

t0 = time.perf_counter()
with ProcessPoolExecutor(max_workers=3) as pool:
    futures = [pool.submit(run_extractor, ext, texts) for ext in DEFAULT_EXTRACTORS]
    extraction_list = [f.result() for f in futures]
wall_extract = time.perf_counter() - t0
extraction = {r.name: r for r in extraction_list}

for r in extraction_list:
    print(f'{r.name:20s} {r.elapsed_seconds:7.2f}s')
sum_extract = sum(r.elapsed_seconds for r in extraction_list)
print(f'\nSuma secuencial: {sum_extract:.2f}s | Pared (paralelo): {wall_extract:.2f}s '
      f'| Speedup: {sum_extract / wall_extract:.2f}x')


In [ ]:
df = raw.copy()
df['keywords'] = extraction['keywords'].values
df['sentiment_polarity'] = extraction['sentiment_polarity'].values
df['entities'] = extraction['entities'].values
df.head(3)


## 2. Vectorización enriquecida y partición train/test

Se construye el mismo espacio de características enriquecido de la Bitácora 02: TF-IDF sobre
`descripción + keywords + entidades` (con `MAX_TFIDF_FEATURES = 10 000`, en `float32`) concatenado con la
polaridad de sentimiento (`scipy.sparse.hstack`). Las categorías se codifican con `LabelEncoder` (cuML exige
etiquetas numéricas). La partición **80/20 estratificada** con `random_state=42` fija un conjunto de prueba
**único e intacto**: el muestreo se aplicará *solo* al entrenamiento, de modo que las 4 variantes (baseline +
3 técnicas) se evalúan sobre exactamente el mismo hold-out.


In [ ]:
import json
import numpy as np
from scipy.sparse import hstack, csr_matrix
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split

def join_entities(cell):
    try:
        return ' '.join(json.loads(cell))
    except Exception:
        return ''

enriched_text = (df['description'].astype(str) + ' '
                 + df['keywords'].astype(str) + ' '
                 + df['entities'].map(join_entities))

vectorizer = TfidfVectorizer(stop_words='english', max_features=MAX_TFIDF_FEATURES, dtype=np.float32)
X_tfidf = vectorizer.fit_transform(enriched_text)
sentiment = csr_matrix(df['sentiment_polarity'].to_numpy(dtype=np.float32)).T   # (n, 1)
X = hstack([X_tfidf, sentiment], format='csr').astype(np.float32)

encoder = LabelEncoder()
y = encoder.fit_transform(df['category'])
target_names = list(encoder.classes_)
print('X:', X.shape, '| y:', y.shape)
print('Clases:', target_names)


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=y)
print('Train:', X_train.shape, '| Test:', X_test.shape)

import collections
train_counts = collections.Counter(int(v) for v in y_train)
for lbl in sorted(train_counts):
    print(f'{target_names[lbl]:24s} {train_counts[lbl]:6d}')
ratio = max(train_counts.values()) / min(train_counts.values())
print(f'Razon de desbalance (mayoritaria/minoritaria): {ratio:.2f}:1')


## 3. Fase CPU paralela — Muestreo (3 técnicas)

Se aplican las tres técnicas de `sampling_strategies.py` **en paralelo** (una por proceso) sobre el conjunto
de entrenamiento:

- **Submuestreo aleatorio** (`RandomUnderSampler`): descarta ejemplos de las clases mayoritarias hasta igualar
  a la minoritaria. Barato pero pierde datos.
- **Sobremuestreo con réplicas** (`RandomOverSampler`): duplica ejemplos de las minoritarias hasta igualar a la
  mayoritaria. Conserva todo pero replica literalmente.
- **Sobremuestreo SMOTE** (`SMOTE`): sintetiza ejemplos nuevos interpolando entre cada muestra minoritaria y sus
  vecinos *k*-NN, sin duplicar.

A diferencia de la clasificación (donde solo viajan métricas escalares entre procesos), aquí la **matriz
remuestreada** es la carga que cruza la frontera de procesos: un costo de IPC no despreciable, propio de esta
etapa. El conjunto de prueba **no** se toca.


In [ ]:
from sampling_strategies import DEFAULT_SAMPLING_STRATEGIES, run_sampler

t0 = time.perf_counter()
with ProcessPoolExecutor(max_workers=3) as pool:
    futures = [pool.submit(run_sampler, s, X_train, y_train) for s in DEFAULT_SAMPLING_STRATEGIES]
    sampling_results = [f.result() for f in futures]
wall_sampling = time.perf_counter() - t0

for r in sampling_results:
    dist = {target_names[k]: v for k, v in r.class_distribution.items()}
    print(f'{r.name:26s} {r.elapsed_seconds:7.2f}s | n={r.n_samples:6d} | {dist}')
sum_sampling = sum(r.elapsed_seconds for r in sampling_results)
print(f'\nSuma secuencial: {sum_sampling:.2f}s | Pared (paralelo): {wall_sampling:.2f}s '
      f'| Speedup: {sum_sampling / wall_sampling:.2f}x')


In [ ]:
# Conjuntos a clasificar: baseline (sin muestreo) + las 3 tecnicas.
datasets = {'Baseline': (X_train, y_train)}
for r in sampling_results:
    datasets[r.name] = (r.X_resampled, r.y_resampled)
list(datasets.keys())


### Distribución de clases y tiempos del muestreo


In [ ]:
import matplotlib.pyplot as plt

labels = list(range(len(target_names)))
techniques = list(datasets.keys())
width = 0.8 / len(techniques)

fig, ax = plt.subplots(figsize=(10, 5))
for i, tech in enumerate(techniques):
    _, yy = datasets[tech]
    cnt = collections.Counter(int(v) for v in np.asarray(yy).ravel().tolist())
    vals = [cnt.get(l, 0) for l in labels]
    ax.bar([x + i * width for x in labels], vals, width, label=tech)
ax.set_xticks([l + width * (len(techniques) - 1) / 2 for l in labels])
ax.set_xticklabels(target_names, rotation=20, ha='right')
ax.set_ylabel('Ejemplos de entrenamiento')
ax.set_title('Distribución de clases por técnica de muestreo')
ax.legend()
fig.tight_layout()
fig.savefig(FIG_DIR / 'cad-actividad4-fig2-distribucion-muestreo.png', dpi=150)
plt.show()


In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
names = [r.name for r in sampling_results]
times = [r.elapsed_seconds for r in sampling_results]
ax.bar(names, times, color='#4C78A8')
ax.axhline(wall_sampling, color='crimson', ls='--',
           label=f'Tiempo de pared (paralelo) = {wall_sampling:.1f}s')
ax.set_ylabel('Segundos')
ax.set_title('Muestreo en paralelo — tiempo por técnica vs. pared')
ax.legend()
plt.xticks(rotation=15, ha='right')
fig.tight_layout()
fig.savefig(FIG_DIR / 'cad-actividad4-fig3-tiempos-muestreo.png', dpi=150)
plt.show()


## 4. Fase GPU secuencial — Clasificación (RF, LogReg, SVM)

Para cada conjunto (baseline + 3 muestreos) se entrenan los **tres** clasificadores cuML **en secuencia**
(una GPU es un único dispositivo; no hay análogo de "un núcleo por algoritmo"). `Random Forest` de cuML no
acepta entrada dispersa, así que su matriz se densifica en `float32` solo para él. Cada corrida reporta
accuracy, **balanced accuracy**, **macro-F1**, tiempos y guarda su matriz de confusión.


In [ ]:
from classification_models_gpu import DEFAULT_GPU_STRATEGIES, run_gpu_classifiers

all_results = []
gpu_wall = {}
for tag, (Xtr, ytr) in datasets.items():
    t0 = time.perf_counter()
    res = run_gpu_classifiers(DEFAULT_GPU_STRATEGIES, Xtr, ytr, X_test, y_test,
                              target_names, MATRIX_DIR, tag=tag)
    gpu_wall[tag] = time.perf_counter() - t0
    all_results.extend(res)

print('Tiempo de pared GPU por conjunto (s):', {k: round(v, 2) for k, v in gpu_wall.items()})


## 5. Resultados y discusiones


In [ ]:
rows = [{
    'Muestreo': r.sampling,
    'Algoritmo': r.name,
    'Accuracy': round(r.accuracy, 4),
    'Balanced acc': round(r.balanced_accuracy, 4),
    'Macro-F1': round(r.macro_f1, 4),
    'Tiempo (s)': round(r.total_seconds, 2),
} for r in all_results]
tabla = pd.DataFrame(rows)
tabla.to_csv(RESULTS_DIR / 'comparativa_muestreo.csv', index=False)
try:
    (RESULTS_DIR / 'comparativa_muestreo.md').write_text(tabla.to_markdown(index=False))
except Exception as e:
    print('to_markdown no disponible:', e)
tabla


In [ ]:
# Heatmap de Macro-F1: técnica de muestreo (fila) x algoritmo (columna).
order = ['Baseline', 'Submuestreo aleatorio', 'Sobremuestreo replicas', 'Sobremuestreo SMOTE']
pivot = tabla.pivot(index='Muestreo', columns='Algoritmo', values='Macro-F1')
pivot = pivot.reindex([o for o in order if o in pivot.index])

fig, ax = plt.subplots(figsize=(8, 5))
im = ax.imshow(pivot.values, cmap='YlGn', aspect='auto',
               vmin=float(np.nanmin(pivot.values)), vmax=1.0)
ax.set_xticks(range(len(pivot.columns))); ax.set_xticklabels(pivot.columns)
ax.set_yticks(range(len(pivot.index)));   ax.set_yticklabels(pivot.index)
for i in range(pivot.shape[0]):
    for j in range(pivot.shape[1]):
        ax.text(j, i, f'{pivot.values[i, j]:.3f}', ha='center', va='center')
ax.set_title('Macro-F1 por técnica de muestreo y algoritmo')
fig.colorbar(im, ax=ax, shrink=0.8)
fig.tight_layout()
fig.savefig(FIG_DIR / 'cad-actividad4-fig4-heatmap-macrof1.png', dpi=150)
plt.show()


In [ ]:
# Accuracy vs Macro-F1 (promedio sobre los 3 algoritmos) por técnica de muestreo:
# la brecha accuracy - macroF1 mide el sesgo hacia la clase mayoritaria.
g = tabla.groupby('Muestreo')[['Accuracy', 'Macro-F1']].mean()
g = g.reindex([o for o in order if o in g.index])

x = np.arange(len(g.index)); w = 0.38
fig, ax = plt.subplots(figsize=(9, 5))
ax.bar(x - w/2, g['Accuracy'], w, label='Accuracy', color='#4C78A8')
ax.bar(x + w/2, g['Macro-F1'], w, label='Macro-F1', color='#F58518')
ax.set_xticks(x); ax.set_xticklabels(g.index, rotation=15, ha='right')
ax.set_ylabel('Métrica (promedio de los 3 algoritmos)')
ax.set_ylim(0, 1.0)
ax.set_title('Accuracy vs. Macro-F1 por técnica de muestreo')
ax.legend()
fig.tight_layout()
fig.savefig(FIG_DIR / 'cad-actividad4-fig5-accuracy-vs-f1.png', dpi=150)
plt.show()


In [ ]:
# Diagrama conceptual del pipeline: CPU paralelo -> CPU paralelo -> GPU secuencial.
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch

fig, ax = plt.subplots(figsize=(12, 5.5))
ax.set_xlim(0, 12); ax.set_ylim(0, 6); ax.axis('off')

def box(x, y, w, h, text, color):
    ax.add_patch(FancyBboxPatch((x, y), w, h, boxstyle='round,pad=0.02',
                                fc=color, ec='#333333', lw=1.2))
    ax.text(x + w/2, y + h/2, text, ha='center', va='center', fontsize=8.5)

CPU = '#BBD7EC'; GPU = '#F6C7A9'
ax.text(1.85, 5.5, 'Fase 1 · CPU en paralelo\nExtracción', ha='center', weight='bold', fontsize=9)
box(0.2, 4.1, 3.3, 0.7, 'Núcleo 1 · Keywords (TF-IDF)', CPU)
box(0.2, 3.2, 3.3, 0.7, 'Núcleo 2 · Sentimiento (TextBlob)', CPU)
box(0.2, 2.3, 3.3, 0.7, 'Núcleo 3 · Entidades (spaCy)', CPU)

ax.text(5.9, 5.5, 'Fase 2 · CPU en paralelo\nMuestreo', ha='center', weight='bold', fontsize=9)
box(4.2, 4.1, 3.3, 0.7, 'Núcleo 1 · Submuestreo', CPU)
box(4.2, 3.2, 3.3, 0.7, 'Núcleo 2 · Réplicas', CPU)
box(4.2, 2.3, 3.3, 0.7, 'Núcleo 3 · SMOTE', CPU)

ax.text(10.0, 5.5, 'Fase 3 · GPU en secuencia\nClasificación', ha='center', weight='bold', fontsize=9)
box(8.3, 4.1, 3.4, 0.7, 'Random Forest', GPU)
box(8.3, 3.2, 3.4, 0.7, 'Regresión Logística', GPU)
box(8.3, 2.3, 3.4, 0.7, 'SVM (RBF)', GPU)

for y in (4.45, 3.55, 2.65):
    ax.add_patch(FancyArrowPatch((3.5, y), (4.2, y), arrowstyle='->', mutation_scale=12, color='#555'))
ax.add_patch(FancyArrowPatch((7.5, 3.55), (8.3, 4.45), arrowstyle='->', mutation_scale=14, color='#555'))
ax.add_patch(FancyArrowPatch((7.5, 3.55), (8.3, 3.55), arrowstyle='->', mutation_scale=14, color='#555'))
ax.add_patch(FancyArrowPatch((7.5, 3.55), (8.3, 2.65), arrowstyle='->', mutation_scale=14, color='#555'))
ax.text(9.98, 1.9, '(secuencial: una GPU)', ha='center', fontsize=7.5, style='italic', color='#555')

fig.savefig(FIG_DIR / 'cad-actividad4-fig1-pipeline.png', dpi=150, bbox_inches='tight')
plt.show()


### Matrices de confusión destacadas

Las 12 matrices (4 conjuntos × 3 algoritmos) quedan en `resultados_matrices/`. El contraste más revelador es el
del **Random Forest**: `Baseline` vs. `Submuestreo aleatorio`. En el *baseline*, el RF vuelca gran parte de las
clases minoritarias hacia *Household* (la mayoritaria); con submuestreo, ese sesgo desaparece y el *recall* de
las minoritarias se dispara, a cambio de una caída moderada en *Household*.


## 6. Conclusiones

- **El muestreo importa solo cuando el clasificador es sensible al desbalance.** El único algoritmo perjudicado
  por el desbalance fue el **Random Forest** (baseline macro-F1 0.795, *balanced accuracy* 0.756). Las tres
  técnicas de muestreo lo rescatan: el **submuestreo aleatorio** fue el mejor (macro-F1 0.896, balanced-acc
  0.900: **+10 y +14 pp**), seguido de réplicas (0.890) y SMOTE (0.881). En cambio, `Regresión Logística` y
  `SVM` ya manejaban bien el desbalance (~0.95–0.96) y el muestreo **no** los mejora; el submuestreo incluso
  baja levemente a la Regresión Logística (descarta datos).

- **El accuracy habría ocultado el problema.** En el RF baseline, accuracy (0.795) y balanced-acc (0.756)
  difieren: esa brecha delata el sesgo hacia *Household*. En los modelos lineales, accuracy ≈ balanced-acc ≈
  macro-F1, señal de que no estaban sesgados.

- **En texto disperso de alta dimensión, el submuestreo simple superó a SMOTE.** SMOTE fue la técnica **más
  débil** para el RF (0.881), probablemente porque interpolar vectores TF-IDF dispersos genera ejemplos
  sintéticos poco informativos; balancear descartando (submuestreo) resultó más limpio y, además, el más
  rápido.

- **Paralelismo en CPU (muestreo): el paralelismo no compensó.** Tiempo de pared ≈ 3.2 s frente a una suma
  serial ≈ 2.6 s → *speedup* ≈ **0.8×**. Dos de las tres técnicas son casi instantáneas y el costo lo pone
  SMOTE más el IPC de devolver las matrices remuestreadas; paralelizar tareas de cómputo minúsculo con
  transferencia grande de datos **penaliza** en vez de acelerar.

- **GPU en secuencia.** El conjunto submuestreado (menos filas) es el más barato de clasificar y el
  sobremuestreado el más caro; el `Random Forest` (denso) es el clasificador más costoso. Las 12 corridas
  sumaron ≈ 28 s en la T4.

- **Elección práctica.** La mejor combinación es **SVM** (macro-F1 ≈ 0.957, con o sin muestreo). Si por costo o
  interpretabilidad se prefiere el Random Forest, **debe** balancearse —preferentemente con submuestreo—, que
  recupera ~10 pp de macro-F1 y encima acelera su entrenamiento.
